In [41]:
import numpy as np
import time
from scipy.io import savemat
import scipy
from opencosmorspy import Parameterization, COSMORS
import math
import os

N = 50
delta = 1. / N

polymers = ['1a', '1b', '1c', '1d', '1e', '1f', '1g', '2a', '2b', '2c', '2d', '2e', '3a', '3b', '4a', '4b', '4c', '4d', '4e']
solvents = polymers.copy()
DFT_path = "/Users/justint/Library/CloudStorage/OneDrive-Personal/Desktop/Academic Stuff/Arias Research/Polymers/2025/November_2025/New_polymers/Ethan/DFT"
poly_volumes = {}
for poly_index, poly_prefix in enumerate(polymers):
    # extract volume from cpcm files
    poly_path = os.path.join(DFT_path, f"{poly_prefix}.cpcm")
    with open(poly_path, 'r') as file:
        poly_lines = file.readlines()
    poly_volume = float(poly_lines[13].strip().split()[0])
    poly_volumes[poly_prefix] = poly_volume
    
    for solvent_index, solvent_prefix in enumerate(solvents):
        if solvent_index <= poly_index:
            continue

        # extract volumes from cpcm files
        poly_path = os.path.join(DFT_path, f"{poly_prefix}.cpcm")
        with open(poly_path, 'r') as file:
            poly_lines = file.readlines()
        solvent_path = os.path.join(DFT_path, f"{solvent_prefix}.cpcm")
        with open(solvent_path, 'r') as file:
            solvent_lines = file.readlines()
        poly_volume = float(poly_lines[13].strip().split()[0])
        solvent_volume = float(solvent_lines[13].strip().split()[0])
        v = np.array([poly_volume,solvent_volume]) # Placeholder for volumes of molecules
        poly_volumes[solvent_prefix] = solvent_volume

        print(f"delta: {delta}")
        print(f"Volumes of molecules: {v}")

        def normalize_and_adjust_mol_fractions(mol_fractions, tolerance=1e-16):
            mol_fractions = np.array(mol_fractions, dtype=np.float64)
            total = np.sum(mol_fractions)
            normalized_mol_fractions = mol_fractions / total
            
            # Check if the sum meets the required precision
            sum_normalized = np.sum(normalized_mol_fractions)
            if abs(1.0 - sum_normalized) > tolerance:
                # Adjust the last element to correct the rounding error
                normalized_mol_fractions[-1] += 1.0 - sum_normalized
            return normalized_mol_fractions

        with open(f'indices_files/indices2phases.txt', 'w') as file:
            with open(f'indices_files/input-cosmors-2phases.py', 'w') as file2:
                for ia in range(0, N+1):
                        ib = N - ia
                        if ia+ib == N and ib >= 0:
                            a = ia * delta
                            b = ib * delta
                            sum_v = a / v[0] + b / v[1]
                            am = a / v[0] * 1 / sum_v
                            bm = b / v[1] * 1 / sum_v
                            normalized_mol_fractions = normalize_and_adjust_mol_fractions([am,bm])
                            print(f"{ia:.0f} {ib:.0f}", file=file)
                            print(f"\nx = np.array([{normalized_mol_fractions[0]}, {normalized_mol_fractions[1]}]) \ncrs.add_job(x, T, refst='pure_component') ", file=file2)

        T = 300 # convert to Kelvin
        filename = f'{poly_prefix}_{solvent_prefix}.mat'

        crs = COSMORS(par='default_orca')
        crs.par.calculate_contact_statistics_molecule_properties = True

        # define molecules
        poly_cosmors = os.path.join(DFT_path, f"{poly_prefix}.cosmors")
        solvent_cosmors = os.path.join(DFT_path, f"{solvent_prefix}.cosmors")
        crs.add_molecule([poly_cosmors])
        crs.add_molecule([solvent_cosmors])


        file_path = f'indices_files/input-cosmors-2phases.py'
        with open(file_path, 'r') as file:
            file_contents = file.read()
        exec(file_contents)
        results = crs.calculate()

        file_path = f'indices_files/indices2phases.txt'
        indices = np.loadtxt(file_path, dtype=int)  # Assuming the indices are integers

        maxN = np.max(indices) + 1  # Calculate the maximum value in indices
        lna = np.zeros((maxN, maxN, 2), dtype=float)  # Initialize lna

        for i in range(len(indices)):
            a = indices[i][0]
            b = indices[i][1]

            lnAs = results['enth']['lng'][i]
            lna[a, b, :] = np.array(lnAs)

        lna4 = lna.astype(float)
        lna4[np.isnan(lna)] = 0.0

        mdict = {'matrix': lna}

        # --- Define output directory ---
        save_dir = "matrices_Ethan"
        os.makedirs(save_dir, exist_ok=True)  # Create the folder if it doesn't exist

        # --- Construct full path for the .mat file ---
        filename = f"{poly_prefix}_{solvent_prefix}.mat"
        save_path = os.path.join(save_dir, filename)

        # --- Save only inside matrices_Ethan ---
        mdict = {'matrix': lna4}
        savemat(save_path, mdict)

        print(f"Saved matrix to: {save_path}")


delta: 0.02
Volumes of molecules: [4627.79606387 8346.03743333]
Saved matrix to: matrices_Ethan/1a_1b.mat
delta: 0.02
Volumes of molecules: [4627.79606387 6330.24671231]
Saved matrix to: matrices_Ethan/1a_1c.mat
delta: 0.02
Volumes of molecules: [4627.79606387 5617.20582926]
Saved matrix to: matrices_Ethan/1a_1d.mat
delta: 0.02
Volumes of molecules: [4627.79606387 4222.43598334]
Saved matrix to: matrices_Ethan/1a_1e.mat
delta: 0.02
Volumes of molecules: [4627.79606387 5981.55066029]
Saved matrix to: matrices_Ethan/1a_1f.mat
delta: 0.02
Volumes of molecules: [4627.79606387 6223.21169493]
Saved matrix to: matrices_Ethan/1a_1g.mat
delta: 0.02
Volumes of molecules: [4627.79606387 4401.79846151]
Saved matrix to: matrices_Ethan/1a_2a.mat
delta: 0.02
Volumes of molecules: [4627.79606387 8312.20025165]
Saved matrix to: matrices_Ethan/1a_2b.mat
delta: 0.02
Volumes of molecules: [4627.79606387 8055.73559282]
Saved matrix to: matrices_Ethan/1a_2c.mat
delta: 0.02
Volumes of molecules: [4627.796063

In [42]:
monomer_volumes = {}
for key, value in poly_volumes.items():
    monomer_volumes[key] = value / 4  # Assuming each polymer has 2 monomer units
monomer_volumes


{'1a': 1156.94901596775,
 '1b': 2086.509358332,
 '1c': 1582.56167807625,
 '1d': 1404.30145731375,
 '1e': 1055.60899583625,
 '1f': 1495.3876650725,
 '1g': 1555.80292373125,
 '2a': 1100.44961537875,
 '2b': 2078.0500629125,
 '2c': 2013.93389820525,
 '2d': 2102.711649514,
 '2e': 3167.40086758925,
 '3a': 1431.475475908,
 '3b': 1700.424273081,
 '4a': 1428.051494551,
 '4b': 1809.36842217625,
 '4c': 2194.86450388725,
 '4d': 2328.886907741,
 '4e': 1634.07045348575}

In [26]:
import cvxpy as cp
import numpy as np
import matplotlib.pyplot as plt
import scipy.io
import pandas as pd

def find_optimal_chi_with_constraints(phi, L):
    """
    Finds the optimal symmetric CHI matrix that minimizes ||Phi * Chi - L||_F^2
    subject to Chi is symmetric and the diagonal of Chi is zero.

    Parameters:
    - phi: numpy.ndarray, shape (N, 2)
    - L: numpy.ndarray, shape (N, 2)

    Returns:
    - Chi_opt: numpy.ndarray, shape (2, 2), the optimized CHI matrix
    """
    phi = np.array(phi)
    L = np.array(L)

    Chi = cp.Variable((2, 2), symmetric=True)

    constraints = [
        cp.diag(Chi) == 0  # Zero diagonal
    ]

    objective = cp.Minimize(cp.norm(phi @ Chi - L, 'fro')**2)
    prob = cp.Problem(objective, constraints)
    prob.solve()

    if prob.status not in ["infeasible", "unbounded"]:
        return Chi.value
    else:
        raise ValueError("Optimization problem is infeasible or unbounded.")

# --------- Build χ matrix over all polymers ----------
polymers = ['1a', '1b', '1c', '1d', '1e', '1f', '1g',
            '2a', '2b', '2c', '2d', '2e',
            '3a', '3b',
            '4a', '4b', '4c', '4d', '4e']

solvents = polymers.copy()
n = len(polymers)

# χ matrix: polymer (row) vs solvent (column)
chi_mat = np.full((n, n), np.nan)

for poly_index, poly_prefix in enumerate(polymers):
    for solvent_index, solvent_prefix in enumerate(solvents):
        # Skip lower triangle & diagonal; fill symmetric later
        if solvent_index <= poly_index:
            continue

        mu_data = scipy.io.loadmat(f"matrices_Ethan/{poly_prefix}_{solvent_prefix}.mat")
        mu = mu_data['matrix']

        # indices along diagonal (i + j == N-1)
        N = mu.shape[0]
        mu_incr = np.arange(N)
        i, j = np.meshgrid(mu_incr, mu_incr, indexing='ij')
        i_flat = i.ravel()
        j_flat = j.ravel()
        condition = (i_flat + j_flat) == (N - 1)

        i_filtered = i_flat[condition]
        j_filtered = j_flat[condition]

        mu_nonzero = mu[i_filtered, j_filtered, :]
        mu_step_temp = np.stack((i_filtered, j_filtered), axis=-1) / (N - 1)

        PHI = mu_step_temp
        L = mu_nonzero

        Chi_opt = find_optimal_chi_with_constraints(PHI, L)
        chi_val = Chi_opt[0, 1] / 4.0

        chi_mat[poly_index, solvent_index] = chi_val
        chi_mat[solvent_index, poly_index] = chi_val  # enforce symmetry

# Optionally set diagonal to 0 (self-interaction) or keep NaN
np.fill_diagonal(chi_mat, 0.0)

# --------- Make a nice, colored table with pandas Styler ----------
chi_df = pd.DataFrame(chi_mat, index=polymers, columns=solvents)

def highlight_groups(df):
    """
    Color cells based on leading digit of either polymer
    (row or column name).

    '1' -> red
    '2' -> blue
    '3' -> green
    '4' -> orange-yellow
    """
    styles = pd.DataFrame("", index=df.index, columns=df.columns)

    for r in df.index:
        for c in df.columns:
            groups = {r[0], c[0]}  # leading digits of row & column
            color = ""
            if "1" in groups:
                color = "background-color: #ffcccc"  # light red
            elif "2" in groups:
                color = "background-color: #cce0ff"  # light blue
            elif "3" in groups:
                color = "background-color: #ccffcc"  # light green
            elif "4" in groups:
                color = "background-color: #ffe5b4"  # orange-yellow
            styles.loc[r, c] = color
    return styles

chi_styled = chi_df.style.format("{:.3f}").apply(highlight_groups, axis=None)

chi_df.to_csv("chi_table.csv")



In [44]:
import os
import cvxpy as cp      # still imported in case you use it elsewhere
import numpy as np
import matplotlib.pyplot as plt
import scipy.io
import pandas as pd

# --------- Settings ----------
polymers = ['1a', '1b', '1c', '1d', '1e', '1f', '1g',
            '2a', '2b', '2c', '2d', '2e',
            '3a', '3b',
            '4a', '4b', '4c', '4d', '4e']

solvents = polymers.copy()
n = len(polymers)
n_monomers = 4        # your scaling factor
DFT_path = "/Users/justint/Library/CloudStorage/OneDrive-Personal/Desktop/Academic Stuff/Arias Research/Polymers/2025/November_2025/New_polymers/Ethan/DFT"

# Matrices for mean chi and its standard error
chi_mean_mat = np.full((n, n), np.nan)
chi_err_mat  = np.full((n, n), np.nan)

# --------- Main loop over polymer/solvent pairs ----------
for poly_index, poly_prefix in enumerate(polymers):
    for solvent_index, solvent_prefix in enumerate(solvents):
        # Skip lower triangle & diagonal; fill symmetric later
        if solvent_index <= poly_index:
            continue

        # --- extract volumes from cpcm files ---
        poly_path = os.path.join(DFT_path, f"{poly_prefix}.cpcm")
        solvent_path = os.path.join(DFT_path, f"{solvent_prefix}.cpcm")

        with open(poly_path, 'r') as file:
            poly_lines = file.readlines()
        with open(solvent_path, 'r') as file:
            solvent_lines = file.readlines()

        poly_volume = float(poly_lines[13].strip().split()[0])
        solvent_volume = float(solvent_lines[13].strip().split()[0])

        # v[0] = polymer molar volume, v[1] = solvent molar volume
        v = np.array([poly_volume, solvent_volume], dtype=float)

        # --- load μ matrix for this pair ---
        mu_data = scipy.io.loadmat(f"matrices_Ethan/{poly_prefix}_{solvent_prefix}.mat")
        mu = mu_data['matrix']   # shape (N, N, 2) presumably

        # indices along diagonal (i + j == N-1)
        N = mu.shape[0]
        mu_incr = np.arange(N)
        i, j = np.meshgrid(mu_incr, mu_incr, indexing='ij')
        i_flat = i.ravel()
        j_flat = j.ravel()
        condition = (i_flat + j_flat) == (N - 1)

        i_filtered = i_flat[condition]
        j_filtered = j_flat[condition]

        # μ_{iα} along that diagonal line, shape (M,2)
        MU = mu[i_filtered, j_filtered, :]

        # φ indices (scaled to [0,1]), shape (M,2)
        # row: point i, columns: components α=0,1
        PHI = np.stack((i_filtered, j_filtered), axis=-1) / (N - 1)

        # ---------- Build factor F_{iα} ----------
        # F_{iα} = [ sum_γ v_α φ_{iγ} / v_γ ] * (1 - φ_{iα})
        # sum_γ φ_{iγ} / v_γ  = (PHI * (1/v)).sum(axis=1)
        inv_v = 1.0 / v                    # shape (2,)
        sum_phi_over_v = (PHI * inv_v).sum(axis=1)   # shape (M,)

        # Now broadcast:
        # factor[i, α] = v[α] * sum_phi_over_v[i] * (1 - PHI[i, α])
        factor = v[None, :] * sum_phi_over_v[:, None] * (1.0 - PHI)  # shape (M,2)

        # ---------- Flatten to a standard linear regression form ----------
        mu_flat = MU.ravel()       # vector of length K = 2*M
        f_flat  = factor.ravel()   # same length

        # Remove any points where factor is zero to avoid degeneracy
        mask = (f_flat != 0.0)
        mu_flat = mu_flat[mask]
        f_flat  = f_flat[mask]

        K = len(mu_flat)
        if K < 2:
            # Not enough data points to fit χ + error
            continue

        # ---------- Least-squares estimate ----------
        # Model: mu_flat ≈ beta * f_flat, where beta = n_monomers * χ
        numer = np.dot(mu_flat, f_flat)
        denom = np.dot(f_flat, f_flat)

        beta_hat = numer / denom           # slope
        chi_hat  = beta_hat / n_monomers   # your χ

        # ---------- Error bars (univariate OLS) ----------
        # Residuals: r_k = μ_k - beta_hat * f_k
        residuals = mu_flat - beta_hat * f_flat

        # Unbiased residual variance σ^2
        # Degrees of freedom: K - 1 (1 parameter: beta)
        sigma2 = np.sum(residuals**2) / (K - 1)

        # Var(beta) = σ^2 / Σ f_k^2
        var_beta = sigma2 / denom

        # Var(chi) = Var(beta) / n_monomers^2
        var_chi = var_beta / (n_monomers**2)
        chi_err = np.sqrt(var_chi)

        # Optionally: 95% CI
        z95 = 1.96
        chi_low  = chi_hat - z95 * chi_err
        chi_high = chi_hat + z95 * chi_err

        # Store symmetric entries
        chi_mean_mat[poly_index, solvent_index] = chi_hat
        chi_mean_mat[solvent_index, poly_index] = chi_hat

        chi_err_mat[poly_index, solvent_index] = chi_err
        chi_err_mat[solvent_index, poly_index] = chi_err

        # (Optional debug print)
        print(f"{poly_prefix}-{solvent_prefix}: "
              f"χ = {chi_hat:.6f} ± {chi_err:.6f} (95% CI: {chi_low:.6f}, {chi_high:.6f})")

# Optionally set diagonal: χ = 0, error = 0
np.fill_diagonal(chi_mean_mat, 0.0)
np.fill_diagonal(chi_err_mat, 0.0)

# --------- Build a "value ± error" string table ----------
chi_str_mat = np.empty((n, n), dtype=object)
for i in range(n):
    for j in range(n):
        val = chi_mean_mat[i, j]
        err = chi_err_mat[i, j]
        if np.isnan(val):
            chi_str_mat[i, j] = ""
        else:
            chi_str_mat[i, j] = f"{val:.6f} ± {err:.6f}"

chi_df_str = pd.DataFrame(chi_str_mat, index=polymers, columns=solvents)

# --------- Coloring by group ----------
def highlight_groups(df):
    """
    Color cells based on leading digit of either polymer
    (row or column name).

    '1' -> red
    '2' -> blue
    '3' -> green
    '4' -> orange-yellow
    """
    styles = pd.DataFrame("", index=df.index, columns=df.columns)

    for r in df.index:
        for c in df.columns:
            groups = {r[0], c[0]}  # leading digits of row & column
            color = ""
            if "1" in groups:
                color = "background-color: #ffcccc"  # light red
            elif "2" in groups:
                color = "background-color: #cce0ff"  # light blue
            elif "3" in groups:
                color = "background-color: #ccffcc"  # light green
            elif "4" in groups:
                color = "background-color: #ffe5b4"  # orange-yellow
            styles.loc[r, c] = color
    return styles

# Styler: we already formatted as strings, so no .format() step
chi_styled = chi_df_str.style.apply(highlight_groups, axis=None)

# Save CSV with "value ± error" entries
chi_df_str.to_csv("chi_table_with_errors.csv")


1a-1b: χ = 0.003744 ± 0.000094 (95% CI: 0.003561, 0.003928)
1a-1c: χ = 0.002700 ± 0.000070 (95% CI: 0.002562, 0.002838)
1a-1d: χ = -0.000253 ± 0.000007 (95% CI: -0.000266, -0.000241)
1a-1e: χ = -0.000005 ± 0.000000 (95% CI: -0.000005, -0.000004)
1a-1f: χ = 0.117866 ± 0.003074 (95% CI: 0.111841, 0.123890)
1a-1g: χ = 0.004447 ± 0.000115 (95% CI: 0.004221, 0.004673)
1a-2a: χ = 0.204375 ± 0.005320 (95% CI: 0.193948, 0.214802)
1a-2b: χ = 0.181052 ± 0.004645 (95% CI: 0.171948, 0.190155)
1a-2c: χ = 0.053467 ± 0.001349 (95% CI: 0.050823, 0.056110)
1a-2d: χ = 0.211046 ± 0.005295 (95% CI: 0.200668, 0.221423)
1a-2e: χ = 0.006624 ± 0.000141 (95% CI: 0.006347, 0.006901)
1a-3a: χ = 0.039447 ± 0.001024 (95% CI: 0.037439, 0.041454)
1a-3b: χ = -0.005164 ± 0.000135 (95% CI: -0.005428, -0.004899)
1a-4a: χ = 0.111919 ± 0.002905 (95% CI: 0.106226, 0.117612)
1a-4b: χ = 1.432167 ± 0.043158 (95% CI: 1.347577, 1.516757)
1a-4c: χ = 0.212176 ± 0.005270 (95% CI: 0.201846, 0.222506)
1a-4d: χ = 0.486894 ± 0.012064 

In [18]:
chi_500 = [chi * 500/4 for chi in chi_list]
print("CHI values scaled to 500 monomer units:", chi_500)

CHI values scaled to 500 monomer units: [np.float64(84.60295933200561), np.float64(50.24371452938707), np.float64(5.327297618789869)]
